In [1]:
!pip install pycaret

# Import

In [2]:
import pandas as pd
import numpy as np
from scipy import stats

from sklearn.impute import SimpleImputer

from sklearn.preprocessing import LabelEncoder, StandardScaler

#from pycaret.classification import setup, compare_models
import pickle
from pycaret.regression import *

In [3]:
df = pd.read_csv("/content/final_internship_data.csv")
print(df.columns)

Index(['User ID', 'User Name', 'Driver Name', 'Car Condition', 'Weather',
       'Traffic Condition', 'key', 'fare_amount', 'pickup_datetime',
       'pickup_longitude', 'pickup_latitude', 'dropoff_longitude',
       'dropoff_latitude', 'passenger_count', 'hour', 'day', 'month',
       'weekday', 'year', 'jfk_dist', 'ewr_dist', 'lga_dist', 'sol_dist',
       'nyc_dist', 'distance', 'bearing'],
      dtype='object')


In [4]:
for column in df.columns:
    print(f"عدد الفئات في العمود '{column}':")
    print(df[column].value_counts())
    print("-" * 30)  # فاصل بين الأعمدة

عدد الفئات في العمود 'User ID':
User ID
KHVrEVlD    1
hp8dBAag    1
OG7FicXv    1
Phl9pRbO    1
Gg8lXxrJ    1
           ..
MKDBAlFt    1
M8lPpDO2    1
4NtMOqM1    1
sJHWgXSJ    1
qGKn4Um5    1
Name: count, Length: 500000, dtype: int64
------------------------------
عدد الفئات في العمود 'User Name':
User Name
Michael Smith       224
Michael Johnson     200
Michael Brown       165
Michael Williams    158
David Smith         152
                   ... 
Cristina Marquez      1
Jack Bowers           1
Wanda Espinoza        1
Ralph Rivers          1
Dillon Jackson        1
Name: count, Length: 221675, dtype: int64
------------------------------
عدد الفئات في العمود 'Driver Name':
Driver Name
Michael Smith          257
David Smith            191
Michael Johnson        183
Michael Williams       167
James Smith            164
                      ... 
Tammy Mayer              1
Juan Wheeler             1
Brandi Franco            1
Mr. Hayden Young MD      1
Lonnie Santana           1
Name: c

# Data Preprocessing

In [5]:
def wrangle(filepath):
    """
    Reads and preprocesses the dataset from the given CSV file.

    Parameters:
    filepath (str): Path to the CSV file.

    Returns:
    pd.DataFrame: Cleaned and preprocessed DataFrame.
    """
    # Read CSV file
    df = pd.read_csv(filepath)

    # Select relevant columns
    # Convert 'pickup_datetime' to datetime format
    df['pickup_datetime'] = pd.to_datetime(df['pickup_datetime'])

    # Extract time-based features
    df['hour'] = df['pickup_datetime'].dt.hour
    df['day'] = df['pickup_datetime'].dt.day
    df['month'] = df['pickup_datetime'].dt.month
    df['year'] = df['pickup_datetime'].dt.year
    df['weekday'] = df['pickup_datetime'].dt.weekday
    # Drop the original 'date of reservation' column
    df.drop(columns=['pickup_datetime'], inplace=True)
    # Define mappings for categorical variables


    # Define mappings for categorical variables
    Traffic_Condition = {
        "Flow Traffic": 0,
        "Dense Traffic": 1,
        "Congested Traffic": 2,


    }
    Weather = {
        "sunny": 1,
        "stormy": 2,
        "rainy": 3,
        "windy": 4,
        "cloudy": 5

    }
    Car_Condition = {
        "Bad": 0,
        "Good ": 1,
        "Very Good": 2,
        "Excellent": 3

    }

        # Apply the mappings to the respective columns
    df['Traffic Condition'] = df['Traffic Condition'].map(Traffic_Condition)
    df['Weather'] = df['Weather'].map(Weather)
    df['Car Condition'] = df['Car Condition'].map(Car_Condition)
    # Handle missing values
    if df.isnull().sum().sum() > 0:
      print("Missing values detected. Handling missing values...")
      df.dropna(inplace=True)

    # Remove duplicate values
    if df.duplicated().sum() > 0:
        df.drop_duplicates(inplace=True)



    return df

# Load and preprocess data
df = wrangle('/content/final_internship_data.csv')
df.head()


Missing values detected. Handling missing values...


,User ID,User Name,Driver Name,Car Condition,Weather,Traffic Condition,key,fare_amount,pickup_longitude,pickup_latitude,...,month,weekday,year,jfk_dist,ewr_dist,lga_dist,sol_dist,nyc_dist,distance,bearing
0,KHVrEVlD,Kimberly Adams,Amy Butler,2.0,4,2,2009-06-15 17:26:21.0000001,4.5,-1.288826,0.710721,...,6,0,2009,20.265840,55.176046,14.342611,34.543548,27.572573,1.030764,-2.918897
1,lPxIuEri,Justin Tapia,Hannah Zimmerman,3.0,5,0,2010-01-05 16:52:16.0000002,16.9,-1.291824,0.710546,...,1,1,2010,44.667679,31.832358,23.130775,15.125872,8.755732,8.450134,-0.375217
2,gsVN8JLS,Elizabeth Lopez,Amanda Jackson,0.0,2,2,2011-08-18 00:35:00.00000049,5.7,-1.291242,0.711418,...,8,3,2011,43.597686,33.712082,19.865289,17.722624,9.847344,1.389525,2.599961
3,9I7kWFgd,Steven Wilson,Amy Horn,2.0,2,0,2012-04-21 04:30:42.0000001,7.7,-1.291319,0.710927,...,4,5,2012,42.642965,32.556289,21.063132,15.738963,7.703421,2.799270,0.133905
4,8QN5ZaGN,Alexander Andrews,Cassandra Larson,0.0,2,2,2010-03-09 07:51:00.000000135,5.3,-1.290987,0.711536,...,3,1,2010,43.329953,39.406828,15.219339,23.732406,15.600745,1.999157,-0.502703


In [6]:
df = df[[ 'Car Condition', 'Weather',
         'Traffic Condition', 'fare_amount',
         'passenger_count', 'hour', 'day', 'month',
         'weekday', 'year', 'jfk_dist', 'ewr_dist', 'lga_dist', 'sol_dist',
         'nyc_dist', 'distance', 'bearing']]


In [7]:
print(df.columns)

Index(['Car Condition', 'Weather', 'Traffic Condition', 'fare_amount',
       'passenger_count', 'hour', 'day', 'month', 'weekday', 'year',
       'jfk_dist', 'ewr_dist', 'lga_dist', 'sol_dist', 'nyc_dist', 'distance',
       'bearing'],
      dtype='object')


In [8]:
print(df.dtypes)

Car Condition        float64
Weather                int64
Traffic Condition      int64
fare_amount          float64
passenger_count        int64
hour                   int32
day                    int32
month                  int32
weekday                int32
year                   int32
jfk_dist             float64
ewr_dist             float64
lga_dist             float64
sol_dist             float64
nyc_dist             float64
distance             float64
bearing              float64
dtype: object


# Check And Handel The Outliers Using (EX : IQR OR Z SCORE )

In [9]:


# نسخ البيانات الأصلية
df_filled_z = df.copy()

# تحديد عتبة Z-score (قابلة للتعديل)
z_threshold = 3

# استبدال القيم الشاذة بأقرب حد مقبول
for col in df_filled_z.select_dtypes(include=['number']).columns:
    # تجاهل القيم المفقودة مؤقتًا عند حساب Z-score
    z_scores = np.abs(stats.zscore(df_filled_z[col], nan_policy='omit'))
    mean_value = df_filled_z[col].mean()
    std_dev = df_filled_z[col].std()

    # حساب الحدود المقبولة
    lower_bound = mean_value - z_threshold * std_dev
    upper_bound = mean_value + z_threshold * std_dev

    # تقييد القيم الشاذة عند الحدود
    df_filled_z[col] = np.where(df_filled_z[col] > upper_bound, upper_bound, df_filled_z[col])
    df_filled_z[col] = np.where(df_filled_z[col] < lower_bound, lower_bound, df_filled_z[col])

print("Dataset after capping outliers using Z-score:\n", df_filled_z)


Dataset after capping outliers using Z-score:
         Car Condition  Weather  Traffic Condition  fare_amount  \
0                 2.0      4.0                2.0          4.5   
1                 3.0      5.0                0.0         16.9   
2                 0.0      2.0                2.0          5.7   
3                 2.0      2.0                0.0          7.7   
4                 0.0      2.0                2.0          5.3   
...               ...      ...                ...          ...   
499994            2.0      4.0                2.0         13.0   
499995            0.0      3.0                1.0          7.0   
499996            2.0      3.0                0.0         13.7   
499997            0.0      3.0                0.0         25.0   
499999            2.0      1.0                0.0          4.9   

        passenger_count  hour   day  month  weekday    year   jfk_dist  \
0                   1.0  17.0  15.0    6.0      0.0  2009.0  20.265840   
1           

#  Feature Engineering ( Feature Selection , Feature Extraction)

In [10]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.decomposition import PCA

# Function to encode categorical features
def encode_categorical_features(df):
    encoder = LabelEncoder()
    df_encoded = df.copy()

    for col in df_encoded.select_dtypes(include='object'):
        df_encoded[col] = encoder.fit_transform(df_encoded[col].astype(str))

    return df_encoded

# Function to scale numerical features
def scale_numerical_features(df):
    scaler = StandardScaler()
    df_scaled = df.copy()

    for col in df_scaled.select_dtypes(include='number'):
        df_scaled[col] = scaler.fit_transform(df_scaled[col].values.reshape(-1, 1))

    return df_scaled



# Apply encoding, scaling, and PCA
df_encoded = encode_categorical_features(df_filled_z)
#f_scaled = scale_numerical_features(df_encoded)


print("Original DataFrame:")
print(df)
print("\nEncoded DataFrame:")
print(df_encoded)





Original DataFrame:
        Car Condition  Weather  Traffic Condition  fare_amount  \
0                 2.0        4                  2          4.5   
1                 3.0        5                  0         16.9   
2                 0.0        2                  2          5.7   
3                 2.0        2                  0          7.7   
4                 0.0        2                  2          5.3   
...               ...      ...                ...          ...   
499994            2.0        4                  2         13.0   
499995            0.0        3                  1          7.0   
499996            2.0        3                  0         13.7   
499997            0.0        3                  0         25.0   
499999            2.0        1                  0          4.9   

        passenger_count  hour  day  month  weekday  year   jfk_dist  \
0                     1    17   15      6        0  2009  20.265840   
1                     1    16    5      1    

# Train Test Split Modeling And Accuracy Calculation


In [11]:
import numpy as np
import pandas as pd
from pycaret.regression import *
# تهيئة بيئة PyCaret
regression_setup = setup(
    data=df_filled_z,
    target="fare_amount",  # المتغير المستهدف
    train_size=0.8,   # نسبة التدريب 80%
    session_id=42,    # للحصول على نفس النتائج في كل مرة
    normalize=True    # تفعيل مقياس البيانات
)
best_model = compare_models()


,Description,Value
0,Session id,42
1,Target,fare_amount
2,Target type,Regression
3,Original data shape,"(375028, 17)"
4,Transformed data shape,"(375028, 17)"
5,Transformed train set shape,"(300022, 17)"
6,Transformed test set shape,"(75006, 17)"
7,Numeric features,16
8,Preprocess,True
9,Imputation type,simple


,,
,,
Initiated,. . . . . . . . . . . . . . . . . .,15:11:58
Status,. . . . . . . . . . . . . . . . . .,Fitting 10 Folds
Estimator,. . . . . . . . . . . . . . . . . .,Random Forest Regressor


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
dt,Decision Tree Regressor,2.4262,22.2141,4.7123,0.6632,0.3261,0.2660,6.3300
lr,Linear Regression,4.5430,43.4152,6.5889,0.3421,0.4672,0.5354,1.0530
br,Bayesian Ridge,4.5431,43.4152,6.5889,0.3421,0.4672,0.5354,0.3110
ridge,Ridge Regression,4.7405,45.7477,6.7636,0.3068,0.4827,0.5574,0.2430
huber,Huber Regressor,4.3276,49.2536,7.0179,0.2536,0.4547,0.4597,4.5430
omp,Orthogonal Matching Pursuit,5.5213,64.9845,8.0611,0.0153,0.5466,0.6268,0.2420
en,Elastic Net,5.5477,65.3609,8.0844,0.0096,0.5492,0.6314,0.2490
lasso,Lasso Regression,5.5818,65.9844,8.1229,0.0002,0.5526,0.6356,0.2520
llar,Lasso Least Angle Regression,5.5818,65.9844,8.1229,0.0002,0.5526,0.6356,0.2400
knn,K Neighbors Regressor,5.7277,70.2410,8.3808,-0.0643,0.5739,0.6304,35.6270


Processing:   0%|          | 0/81 [00:00<?, ?it/s]

,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
xgboost,Extreme Gradient Boosting,1.6344,10.1807,3.1902,0.8457,0.2293,0.1854,2.5070
lightgbm,Light Gradient Boosting Machine,1.6673,10.2598,3.2025,0.8445,0.2291,0.1870,7.4130
rf,Random Forest Regressor,1.7177,10.5704,3.2506,0.8398,0.2374,0.1989,415.7130
et,Extra Trees Regressor,1.8244,11.1156,3.3335,0.8315,0.2433,0.2116,151.1780
gbr,Gradient Boosting Regressor,1.8214,11.2821,3.3584,0.8290,0.2395,0.2043,88.1950
dt,Decision Tree Regressor,2.4262,22.2141,4.7123,0.6632,0.3261,0.2660,6.3300
lr,Linear Regression,4.5430,43.4152,6.5889,0.3421,0.4672,0.5354,1.0530
br,Bayesian Ridge,4.5431,43.4152,6.5889,0.3421,0.4672,0.5354,0.3110
ada,AdaBoost Regressor,5.3669,45.3698,6.4809,0.3160,0.5381,0.7787,15.8320
ridge,Ridge Regression,4.7405,45.7477,6.7636,0.3068,0.4827,0.5574,0.2430


In [12]:
model = create_model('xgboost')  # نموذج xgboost


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,1.6025,9.7960,3.1299,0.8540,0.2233,0.1721
1,1.6485,10.2006,3.1938,0.8473,0.2269,0.1729
2,1.6466,10.5648,3.2504,0.8376,0.2300,0.2067
3,1.6115,9.6826,3.1117,0.8554,0.2259,0.1755
4,1.6138,9.6478,3.1061,0.8512,0.2304,0.2369
5,1.6163,9.9136,3.1486,0.8514,0.2289,0.1774
6,1.6524,10.5940,3.2548,0.8359,0.2325,0.1786
7,1.6598,10.5695,3.2511,0.8393,0.2342,0.1811
8,1.6468,10.3209,3.2126,0.8437,0.2310,0.1774


Processing:   0%|          | 0/4 [00:00<?, ?it/s]

In [13]:
tuned_model = tune_model(model)
evaluate_model(tuned_model)


,MAE,MSE,RMSE,R2,RMSLE,MAPE
Fold,,,,,,
0,1.6759,10.1095,3.1795,0.8493,0.2279,0.1830
1,1.7174,10.5345,3.2457,0.8423,0.2302,0.1822
2,1.7123,10.8163,3.2888,0.8338,0.2330,0.2117
3,1.6852,9.9827,3.1595,0.8509,0.2294,0.1859
4,1.6936,9.8989,3.1462,0.8473,0.2339,0.2485
5,1.6942,10.1904,3.1922,0.8473,0.2327,0.1884
6,1.7189,10.7499,3.2787,0.8335,0.2349,0.1882
7,1.7323,10.8994,3.3014,0.8343,0.2378,0.1910
8,1.7225,10.6700,3.2665,0.8384,0.2357,0.1886


Processing:   0%|          | 0/7 [00:00<?, ?it/s]

Fitting 10 folds for each of 10 candidates, totalling 100 fits


Original model was better than the tuned model, hence it will be returned. NOTE: The display metrics are for the tuned model (not the original one).


interactive(children=(ToggleButtons(description='Plot Type:', icons=('',), options=(('Pipeline Plot', 'pipelin…

In [14]:
predictions = predict_model(tuned_model)
print(predictions.head())


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE
0,Extreme Gradient Boosting,1.6341,10.1037,3.1786,0.8474,0.2312,0.2114


        Car Condition  Weather  Traffic Condition  passenger_count  hour  \
290227            3.0      3.0                1.0              1.0  19.0   
221799            3.0      3.0                0.0              1.0  18.0   
246833            0.0      2.0                0.0              1.0  13.0   
328328            0.0      1.0                1.0              1.0   0.0   
18394             3.0      5.0                2.0              2.0  22.0   

         day  month  weekday    year   jfk_dist   ewr_dist   lga_dist  \
290227  15.0    5.0      4.0  2009.0  39.839378  32.164566  23.525682   
221799  13.0    9.0      0.0  2010.0  43.981129  30.499214  22.948063   
246833  22.0    1.0      6.0  2012.0  44.085133  28.134464  25.576107   
328328  26.0    7.0      1.0  2011.0  41.293118  29.885773  23.983637   
18394   26.0   10.0      2.0  2011.0  45.033787  36.863377  17.985613   

         sol_dist   nyc_dist  distance   bearing  fare_amount  \
290227  13.308780   7.997581  7.292600 

In [15]:
predictions.head(10)

,Car Condition,Weather,Traffic Condition,passenger_count,hour,day,month,weekday,year,jfk_dist,ewr_dist,lga_dist,sol_dist,nyc_dist,distance,bearing,fare_amount,prediction_label
290227,3.0,3.0,1.0,1.0,19.0,15.0,5.0,4.0,2009.0,39.839378,32.164566,23.525682,13.308780,7.997581,7.292600,2.968070,18.500000,19.431591
221799,3.0,3.0,0.0,1.0,18.0,13.0,9.0,0.0,2010.0,43.981129,30.499214,22.948063,14.102795,6.537919,2.169850,2.848146,41.369801,6.960011
246833,0.0,2.0,0.0,1.0,13.0,22.0,1.0,6.0,2012.0,44.085133,28.134464,25.576107,10.956804,4.036125,3.231483,-0.124885,6.900000,7.850439
328328,0.0,1.0,1.0,1.0,0.0,26.0,7.0,1.0,2011.0,41.293118,29.885773,23.983637,11.250714,2.807358,1.900940,2.557924,6.100000,6.215448
18394,3.0,5.0,2.0,2.0,22.0,26.0,10.0,2.0,2011.0,45.033787,36.863377,17.985613,21.892620,14.108022,2.916279,-0.495123,8.900000,7.049281
328126,2.0,5.0,1.0,5.0,15.0,13.0,12.0,0.0,2010.0,37.203163,45.096664,9.894919,27.430231,19.174089,9.380241,1.708472,24.100000,30.525393
432682,0.0,5.0,2.0,3.0,7.0,5.0,2.0,1.0,2013.0,40.940372,39.913551,13.896317,23.185678,14.814192,2.099581,2.639265,7.000000,6.648122
495366,0.0,5.0,1.0,2.0,22.0,22.0,1.0,4.0,2010.0,43.476345,41.304382,13.875932,25.770790,17.607063,0.885347,-0.005420,4.500000,4.623110
153395,3.0,5.0,2.0,1.0,15.0,12.0,12.0,0.0,2011.0,41.855408,28.971714,24.862629,10.413842,2.122456,0.941359,-0.748459,4.500000,6.117116
281684,3.0,2.0,2.0,2.0,19.0,27.0,4.0,6.0,2014.0,43.606739,29.121124,24.248028,12.101710,4.573017,0.703359,-0.372336,4.000000,4.607075


In [16]:
save_model(tuned_model, 'best_regression_model')

# لاسترجاع النموذج لاحقًا
loaded_model = load_model('best_regression_model')


Transformation Pipeline and Model Successfully Saved
Transformation Pipeline and Model Successfully Loaded
